# CLIP 기반 Scene Frame 추출 노트북

현재 최종 인스타그램 overlay 파이프라인과 같은 CLIP embedding distance 방식으로 scene을 나누고, 각 scene의 대표 프레임만 추출하는 노트북입니다.

- 모델 예측 없음
- overlay 영상 생성 없음
- 라벨링 후보 이미지 추출 전용
- 결과 저장 위치: `data/outputs_clip_frame_extraction/`



In [4]:
# 필요한 라이브러리와 기존 파이프라인 함수를 불러옵니다.
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import shotguide_batch_video_inference as base
import shotguide_instagram_overlay_pipeline as pipe

# 노트북 실행 위치를 프로젝트 루트로 사용합니다.
# 새로 프레임을 추출할 인스타그램 Reels 링크를 여기에 추가합니다.
# 첫 번째 값은 기존 데이터셋처럼 영상 고유 번호(item_id)로 사용됩니다.
EXTRACTION_ITEMS = [
    ("0204", "https://www.instagram.com/reels/DYzpeKtoBUp/"),
    ("0205", "https://www.instagram.com/reels/DYzR-nOzk17/"),
    ("0206", "https://www.instagram.com/reels/DYhB5z8RoUs/"),
    ("0207", "https://www.instagram.com/reels/DYyRx6jhd-J/"),
    ("0208", "https://www.instagram.com/reels/DYjOewlTGko/"),
    ("0209", "https://www.instagram.com/reels/DYzSORLBQJ3/"),
    ("0210", "https://www.instagram.com/reels/DYybTo_Sn-u/"),
    ("0211", "https://www.instagram.com/reels/DYzHCCSyeCz/"),
    ("0212", "https://www.instagram.com/reels/DYy6Q8ivn02/"),
    ("0213", "https://www.instagram.com/reels/DYKRKcLzETX/"),
    ("0214", "https://www.instagram.com/reels/DYyffqLszrm/"),
    ("0215", "https://www.instagram.com/reels/DYzACZMxAmO/"),
    ("0216", "https://www.instagram.com/reels/DYyjoqFxE9q/"),
    ("0217", "https://www.instagram.com/reels/DYzCVPgheT9/"),
    ("0218", "https://www.instagram.com/reels/DXJ187riehG/"),
    ("0219", "https://www.instagram.com/reels/DYojr_bJ1a6/"),
    ("0220", "https://www.instagram.com/reels/DYzCmO7vX41/"),
    ("0221", "https://www.instagram.com/reels/DYwN5gtz2pO/"),
]

# 추출 결과 저장 폴더입니다.
OUTPUT_ROOT = ROOT / "data/outputs_clip_frame_extraction"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# scene 하나당 추출할 대표 프레임 수입니다.
NUM_FRAME_SAMPLES = 1

base.DEVICE



device(type='cpu')

## 추출 함수

링크 하나에 대해 다음 작업만 수행합니다.

1. 인스타그램 영상 다운로드
2. CLIP embedding distance 기반 scene detection
3. scene별 대표 프레임 추출
4. 기존 데이터셋과 같은 파일명으로 이미지 저장
5. 메타데이터 CSV 저장



In [5]:
def save_scene_frames_with_dataset_names(video_path: Path, scene_df: pd.DataFrame, output_dir: Path, item_id: str, num_samples=1):
    # 기존 데이터셋과 같은 파일명 형식으로 이미지만 저장합니다.
    # 예: 0131_cut_001.jpg, 0131_cut_002.jpg
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    saved_paths = []
    cut_count = 1

    for _, scene in scene_df.iterrows():
        frame_indices = base.sample_frame_indices(
            int(scene.start_frame),
            int(scene.end_frame),
            num_samples=num_samples,
        )

        for frame_idx in frame_indices:
            frame = base.read_frame_at(cap, frame_idx)
            if frame is None:
                continue

            frame_path = output_dir / f"{item_id}_cut_{cut_count:03d}.jpg"
            cv2.imwrite(str(frame_path), frame)
            saved_paths.append(frame_path)
            cut_count += 1

    cap.release()
    return saved_paths


def extract_scene_frames_only(item_id: str, url: str, download_index: int, clip_model, clip_preprocess):
    # 기존 파이프라인의 다운로드 함수를 사용해 인스타그램 영상을 저장합니다.
    video_path = pipe.download_instagram_video(url, download_index)
    shortcode = pipe.get_shortcode(url)

    # 각 영상별 폴더 바로 아래에 이미지를 저장합니다.
    video_folder_name = f"{item_id}_{shortcode}"
    video_output_dir = OUTPUT_ROOT / video_folder_name
    video_output_dir.mkdir(parents=True, exist_ok=True)

    # 현재 최종 파이프라인과 같은 CLIP distance 방식으로 scene을 나눕니다.
    scene_df, _distance_df, threshold, info = pipe.detect_scenes_clip_distance(
        video_path,
        clip_model,
        clip_preprocess,
    )

    # CSV나 중간 폴더를 만들지 않고, 이미지 파일만 저장합니다.
    saved_paths = save_scene_frames_with_dataset_names(
        video_path,
        scene_df,
        video_output_dir,
        item_id=item_id,
        num_samples=NUM_FRAME_SAMPLES,
    )

    return {
        "item_id": item_id,
        "url": url,
        "shortcode": shortcode,
        "video_path": str(video_path.resolve()),
        "output_dir": str(video_output_dir.resolve()),
        "scene_count": int(len(scene_df)),
        "duration_sec": float(info["duration_sec"]),
        "threshold": float(threshold),
        "num_extracted_frames": len(saved_paths),
    }



## 실행

`EXTRACTION_ITEMS`에 입력한 `(영상 번호, 링크)`를 순서대로 처리합니다. 결과 이미지는 각 영상 폴더 바로 아래에 저장됩니다.



In [6]:
# scene detection에 필요한 CLIP backbone을 불러옵니다.
# 이 노트북에서는 예측을 하지 않고, 현재 파이프라인 기준으로 이미지만 추출합니다.
clip_model, clip_preprocess, _, _ = base.load_models()

summary_rows = []

for download_index, (item_id, url) in enumerate(EXTRACTION_ITEMS, start=1):
    print("=" * 80)
    print(f"[{item_id}] extracting frames:", url)
    result = extract_scene_frames_only(
        item_id,
        url,
        download_index,
        clip_model,
        clip_preprocess,
    )
    summary_rows.append(result)
    print(f"saved images: {result['num_extracted_frames']} -> {result['output_dir']}")

pd.DataFrame(summary_rows)



c:\Users\eunpa\anaconda3\envs\sy\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[0204] extracting frames: https://www.instagram.com/reels/DYzpeKtoBUp/
[Instagram] Extracting URL: https://www.instagram.com/reels/DYzpeKtoBUp/
[Instagram] DYzpeKtoBUp: Setting up session
[Instagram] DYzpeKtoBUp: Downloading JSON metadata
[info] DYzpeKtoBUp: Downloading 1 format(s): 1
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_001_DYzpeKtoBUp.mp4
[download] 100% of   11.98MiB in 00:00:00 at 17.78MiB/s  


saved images: 16 -> C:\Temp\shot-classification\outputs_clip_frame_extraction\0204_DYzpeKtoBUp
[0205] extracting frames: https://www.instagram.com/reels/DYzR-nOzk17/
[Instagram] Extracting URL: https://www.instagram.com/reels/DYzR-nOzk17/
[Instagram] DYzR-nOzk17: Setting up session
[Instagram] DYzR-nOzk17: Downloading JSON metadata
[info] DYzR-nOzk17: Downloading 1 format(s): 1
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_002_DYzR-nOzk17.mp4
[download] 100% of    1.58MiB in 00:00:00 at 11.08MiB/s  


saved images: 2 -> C:\Temp\shot-classification\outputs_clip_frame_extraction\0205_DYzR-nOzk17
[0206] extracting frames: https://www.instagram.com/reels/DYhB5z8RoUs/
[Instagram] Extracting URL: https://www.instagram.com/reels/DYhB5z8RoUs/
[Instagram] DYhB5z8RoUs: Setting up session
[Instagram] DYhB5z8RoUs: Downloading JSON metadata
[info] DYhB5z8RoUs: Downloading 1 format(s): 1
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_003_DYhB5z8RoUs.mp4
[download] 100% of   12.79MiB in 00:00:47 at 276.21KiB/s    


saved images: 31 -> C:\Temp\shot-classification\outputs_clip_frame_extraction\0206_DYhB5z8RoUs
[0207] extracting frames: https://www.instagram.com/reels/DYyRx6jhd-J/
[Instagram] Extracting URL: https://www.instagram.com/reels/DYyRx6jhd-J/
[Instagram] DYyRx6jhd-J: Setting up session
[Instagram] DYyRx6jhd-J: Downloading JSON metadata
[info] DYyRx6jhd-J: Downloading 1 format(s): 3
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_004_DYyRx6jhd-J.mp4
[download] 100% of    6.86MiB in 00:00:11 at 595.17KiB/s 


...

saved images: 12 -> C:\Temp\shot-classification\outputs_clip_frame_extraction\0218_DXJ187riehG
[0219] extracting frames: https://www.instagram.com/reels/DYojr_bJ1a6/
[Instagram] Extracting URL: https://www.instagram.com/reels/DYojr_bJ1a6/
[Instagram] DYojr_bJ1a6: Setting up session
[Instagram] DYojr_bJ1a6: Downloading JSON metadata
[info] DYojr_bJ1a6: Downloading 1 format(s): 8
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_016_DYojr_bJ1a6.mp4
[download] 100% of    2.54MiB in 00:00:00 at 6.31MiB/s   


saved images: 5 -> C:\Temp\shot-classification\outputs_clip_frame_extraction\0219_DYojr_bJ1a6
[0220] extracting frames: https://www.instagram.com/reels/DYzCmO7vX41/
[Instagram] Extracting URL: https://www.instagram.com/reels/DYzCmO7vX41/
[Instagram] DYzCmO7vX41: Setting up session
[Instagram] DYzCmO7vX41: Downloading JSON metadata
[info] DYzCmO7vX41: Downloading 1 format(s): 3
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_017_DYzCmO7vX41.mp4
[download] 100% of    3.79MiB in 00:00:00 at 12.43MiB/s  


saved images: 11 -> C:\Temp\shot-classification\outputs_clip_frame_extraction\0220_DYzCmO7vX41
[0221] extracting frames: https://www.instagram.com/reels/DYwN5gtz2pO/
[Instagram] Extracting URL: https://www.instagram.com/reels/DYwN5gtz2pO/
[Instagram] DYwN5gtz2pO: Setting up session
[Instagram] DYwN5gtz2pO: Downloading JSON metadata
[info] DYwN5gtz2pO: Downloading 1 format(s): 1
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_018_DYwN5gtz2pO.mp4
[download] 100% of    1.99MiB in 00:00:01 at 1.64MiB/s   


saved images: 6 -> C:\Temp\shot-classification\outputs_clip_frame_extraction\0221_DYwN5gtz2pO


,item_id,url,shortcode,video_path,output_dir,scene_count,duration_sec,threshold,num_extracted_frames
0,0204,https://www.instagram.com/reels/DYzpeKtoBUp/,DYzpeKtoBUp,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,16,44.433333,0.187803,16
1,0205,https://www.instagram.com/reels/DYzR-nOzk17/,DYzR-nOzk17,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,2,34.200000,0.055000,2
2,0206,https://www.instagram.com/reels/DYhB5z8RoUs/,DYhB5z8RoUs,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,31,70.733333,0.076635,31
3,0207,https://www.instagram.com/reels/DYyRx6jhd-J/,DYyRx6jhd-J,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,13,33.133333,0.086259,13
4,0208,https://www.instagram.com/reels/DYjOewlTGko/,DYjOewlTGko,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,2,5.866667,0.055000,2
5,0209,https://www.instagram.com/reels/DYzSORLBQJ3/,DYzSORLBQJ3,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,19,45.366667,0.101148,19
6,0210,https://www.instagram.com/reels/DYybTo_Sn-u/,DYybTo_Sn-u,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,22,45.233333,0.075447,22
7,0211,https://www.instagram.com/reels/DYzHCCSyeCz/,DYzHCCSyeCz,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,9,24.966667,0.197163,9
8,0212,https://www.instagram.com/reels/DYy6Q8ivn02/,DYy6Q8ivn02,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,14,33.200000,0.169437,14
9,0213,https://www.instagram.com/reels/DYKRKcLzETX/,DYKRKcLzETX,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,8,18.000000,0.084628,8


## 결과 확인 방법

실행 후 아래 구조로 이미지만 생성됩니다.

```text
data/outputs_clip_frame_extraction/
  0131_SHORTCODE/
    0131_cut_001.jpg
    0131_cut_002.jpg
    0131_cut_003.jpg
```

추출된 이미지 중 라벨 기준에 맞는 이미지만 `labeled_dataset/`의 적절한 폴더로 복사해서 사용하면 됩니다.

